In [3]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [6]:
output = df["diagnosis"]
output.head()

0    M
1    M
2    M
3    M
4    M
Name: diagnosis, dtype: str

In [7]:
input = df.drop(columns=['diagnosis'])
input.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
X_train, X_test, y_train, y_test = train_test_split(input, output, test_size=0.2)

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
#The error is caused by StandardScaler producing float64 features, 
# while PyTorch’s nn.Linear uses float32 weights by default.

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [16]:
X_train_tensor.shape

torch.Size([455, 30])

In [58]:
import torch.nn as nn

class MySimpleNN(nn.Module):

  def __init__(self, nn_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(nn_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self, features):
    y_pred = self.network(features)
    return y_pred


In [59]:
learning_rate = 0.1
epochs = 50

In [ ]:
# Create the model using the number of input features
model = MySimpleNN(X_train_tensor.shape[1])

# Set up binary cross-entropy loss and SGD to update the model's parameters
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    optimizer.zero_grad()                 # Clear gradients left from the previous step

    y_pred = model(X_train_tensor)        # Run the training features through the model
    loss = criterion(y_pred, y_train_tensor)  # Compare predictions with the true labels

    loss.backward()                       # Calculate gradients for the model parameters
    optimizer.step()                      # Update parameters using those gradients

    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")  # Report loss for this epoch

Epoch: 1, Loss: 0.64185631275177
Epoch: 2, Loss: 0.6229612827301025
Epoch: 3, Loss: 0.6050133109092712
Epoch: 4, Loss: 0.5875389575958252
Epoch: 5, Loss: 0.5705896019935608
Epoch: 6, Loss: 0.55402672290802
Epoch: 7, Loss: 0.5373089909553528
Epoch: 8, Loss: 0.52071613073349
Epoch: 9, Loss: 0.5044743418693542
Epoch: 10, Loss: 0.4886249899864197
Epoch: 11, Loss: 0.4729038178920746
Epoch: 12, Loss: 0.457538902759552
Epoch: 13, Loss: 0.4426240921020508
Epoch: 14, Loss: 0.4279116988182068
Epoch: 15, Loss: 0.41350606083869934
Epoch: 16, Loss: 0.399509996175766
Epoch: 17, Loss: 0.3858796954154968
Epoch: 18, Loss: 0.37259337306022644
Epoch: 19, Loss: 0.3596838116645813
Epoch: 20, Loss: 0.347217321395874
Epoch: 21, Loss: 0.33515986800193787
Epoch: 22, Loss: 0.3235654830932617
Epoch: 23, Loss: 0.3124620318412781
Epoch: 24, Loss: 0.3018370270729065
Epoch: 25, Loss: 0.29169565439224243
Epoch: 26, Loss: 0.2820480167865753
Epoch: 27, Loss: 0.2728779911994934
Epoch: 28, Loss: 0.2641718089580536
Epoch:

In [61]:
#This prints each layer’s weights and biases
for name, param in model.named_parameters():
    print(name, param.detach())

network.0.weight tensor([[ 2.0882e-01,  2.1063e-01,  1.6189e-01,  8.2525e-02,  1.0322e-01,
          1.6698e-01,  9.5051e-02,  8.8138e-02,  1.1021e-01, -1.2093e-01,
         -1.1260e-02,  1.1398e-01,  2.8319e-01,  2.1127e-01, -1.8913e-02,
         -2.9541e-02,  1.1274e-02,  1.6087e-01, -1.3674e-01, -1.1616e-01,
          1.2106e-01,  6.7482e-02,  7.3336e-02,  3.9311e-01,  3.8932e-02,
          1.9908e-01,  2.3692e-01,  3.6147e-01,  2.8032e-01, -2.8768e-02],
        [-2.9221e-01, -1.3437e-01, -3.0444e-01, -2.5048e-01, -1.2260e-01,
         -2.8825e-01, -2.8715e-01, -1.6386e-01,  2.7271e-02,  1.4526e-01,
          5.3747e-04, -1.6701e-04, -4.8555e-02,  5.0515e-02,  1.3630e-01,
          6.1095e-02,  1.6795e-02,  1.6264e-02,  6.9890e-02,  6.0950e-02,
         -2.2106e-01, -1.3995e-02, -2.0503e-01, -1.9253e-01, -1.8623e-01,
         -2.0160e-01, -1.0576e-01, -2.0376e-04, -1.6503e-01,  7.9601e-02],
        [ 1.9573e-01,  1.8243e-01, -8.7278e-02,  8.4259e-02, -5.3268e-02,
          1.6896e-0

In [65]:
# model evaluation
with torch.no_grad():
  y_pred = model(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.9561403393745422


In [ ]:
#%pip install torchinfo

4185.11s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [63]:
from torchinfo import summary

summary(model=model, input_size=X_train_tensor.shape)

Layer (type:depth-idx)                   Output Shape              Param #
MySimpleNN                               [455, 1]                  --
├─Sequential: 1-1                        [455, 1]                  --
│    └─Linear: 2-1                       [455, 3]                  93
│    └─ReLU: 2-2                         [455, 3]                  --
│    └─Linear: 2-3                       [455, 1]                  4
│    └─Sigmoid: 2-4                      [455, 1]                  --
Total params: 97
Trainable params: 97
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.04
Input size (MB): 0.05
Forward/backward pass size (MB): 0.01
Params size (MB): 0.00
Estimated Total Size (MB): 0.07